In [1]:
import pyodbc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

# prep

In [2]:
# Data prep
query = f"""

SELECT
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END AS a,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END AS b
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state
WHERE intragroup = 1
GROUP BY
  CASE WHEN borrower_id < lender_id THEN borrower_id ELSE lender_id END,
  CASE WHEN borrower_id < lender_id THEN lender_id ELSE borrower_id END;


"""

df_edges = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_8724\143588195.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_edges = pd.read_sql_query(query, cnxn)


In [3]:
import pandas as pd

# edges: columns ['a','b']
edges = df_edges.values.tolist()

parent = {}
rank = {}

def find(x):
    if parent.setdefault(x, x) != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(x, y):
    rx, ry = find(x), find(y)
    if rx == ry: return
    rxr, ryr = rank.setdefault(rx,0), rank.setdefault(ry,0)
    if rxr < ryr:
        parent[rx] = ry
    elif rxr > ryr:
        parent[ry] = rx
    else:
        parent[ry] = rx
        rank[rx] = rxr + 1

for a,b in edges:
    union(a,b)

# canonical id = min-LEI per component (stable label)
from collections import defaultdict
comps = defaultdict(list)
for x in parent.keys():
    comps[find(x)].append(x)

canon = {rep: min(members) for rep, members in comps.items()}
mapping = []
for x in parent.keys():
    rep = find(x)
    mapping.append((x, canon[rep]))

df_mapping_cc = pd.DataFrame(mapping, columns=["lei","group_id_synth"]).sort_values(["group_id_synth","lei"])


In [6]:
ust = pd.read_csv('Data\\TreasuryCusip.csv')

In [7]:
treasuries = tuple(ust['ISIN'].unique())

In [8]:
# Data prep
query = f"""

SELECT security_isin
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state f
WHERE security_type IN ('GOVS', 'FIDE')
AND assttp_scty_issr_sector_riad = 'S1311'
AND business_date >= '2023-01-01' 
AND gnlcoll = 'SPEC'
GROUP BY security_isin
HAVING COUNT(DISTINCT security_type) = 2

"""

govs = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_8724\2371169088.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  govs = pd.read_sql_query(query, cnxn)


In [9]:
unique_govs = tuple(govs['security_isin'])

In [10]:
foreign_bg = ('0W1U67PTV5WY3WYWKD79',
'2138008P9NOMBRMROI73',
'213800A9GT65GAES2V60',
'213800G8QEXN34A2YG53',
'213800GVD8L87R18CT98',
'213800IBT39XQ9C4CP71',
'213800NW35DTWHTMX505',
'213800RZ3GCUXMBGYN59',
'2IGI19DL77OX0HC3ZE78',
'4PQUHN3JPFGFNF3BB653',
'4ZHCHI4KYZG2WVRT8631',
'5299007QVIQ7IO64NX37',
'5493000IQQ05Y25L0V92',
'5493000YPN33HF74SN02',
'54930010P7BUGOECPI58',
'5493002XYZZ0CGQ6CB58',
'54930040QPHHWGT1J432',
'54930050SE0SM7CM2G07',
'5493006PWI2H6PX25403',
'5493007VSMFZCPV1NB83',
'5493008GNQHVI377MY19',
'5493009NLZXZGJDOPC94',
'549300BKWHXYEXPV0328',
'549300FH0WJAPEHTIQ77',
'549300HU9EWFS3CNX640',
'549300KP56LL8NKKFL47',
'549300RSY622D5TQWS42',
'549300SXSTGQY3EA1B18',
'549300WDT1HWUMTUW770',
'571474TGEMMWANRLN572',
'724500AAT1DK36059L16',
'9676007O0UF5YB3QPR03',
'BWS7DNS2Z4NPKPNYKL75',
'HV5W8PGLJ127N2SFSM23'
)

In [11]:
euro_area_bg = ('0IKLU6X1B10WK7X42C15',
'1VUV7VQFKUOQSJ21A208',
'2138001YBO7CJNQOEE37',
'21380027LW8AF6I1WA03',
'2138003Z5ZVN16GFYV70',
'2138004VZX8CSGPTDX68',
'21380073P7J4PAD91E29',
'213800AGKVL18YKQSV51',
'213800BWHAS44J2C1B28',
'213800D4LHBCXXEEE235',
'213800DBQIB6VBNU5C64',
'213800FKGFR7Q2ACLS83',
'213800G63T4ER4MSVR22',
'213800HV6TP2I5A6MW58',
'213800I92TAU7I3FP232',
'213800KGF4EFNUQKAT69',
'213800OOQOSULB37T658',
'213800ZIGVOZ992FNQ85',
'222100D7H9VRJEH7DU25',
'222100M2PU043YB7YQ06',
'23B6332KMR0JIZLJG565',
'2534006G7F7F1TFC9T77',
'2534006HF1L4YF10UD91',
'253400N2R0RF14JQ0060',
'2549002MVYWWDX54IB83',
'259400QHDOZWMJ103294',
'259400YLRTOBISHBVX41',
'2W8N8UU78PMDQKZENC08',
'31570010000000036567',
'315700GBLUBZ50S45F53',
'315700GXRKZ452JF2U13',
'3M5E1GQGKL17HI6CPN30',
'3U8WV1YX2VMUHH7Z1Q21',
'48510000156ESYOBV122',
'52965FONQ5NZKP0WZL45',
'52990002O5KK6XOGJ020',
'52990010C4NK412ZL440',
'5299004TE2DYMKEAM814',
'5299005UJX6K7BQKV086',
'52990080NNXXLC14OC65',
'529900AQBND3S6YJLY83',
'529900C214QOT3ZYD838',
'529900C4RSSBWXBSY931',
'529900E1WHT64CB20277',
'529900FWTU88V844B672',
'529900GGYMNGRQTDOO93',
'529900K16YGKC8BES892',
'529900T32UL0CP1FZA06',
'529900UKZBMDBDZIXD62',
'529900VA5CNBWXAONR25',
'549300298FD7AS4PPU70',
'54930056IRBXK0Q1FP96',
'549300685QG7DJS55M76',
'5493007RT80TMKY7TO30',
'549300ABE4K96QOCEH37',
'549300CQ9NLEHMRCU505',
'549300DV870NBWY5W279',
'549300DY78U4CMKNHE48',
'549300DYPOFMXOR7XM56',
'549300FOF121DSRG5867',
'549300FR956J8UJDWQ78',
'549300GOF5A5DHWJLV03',
'549300GRXFI7D6PNEA68',
'549300L7YCATGO57ZE10',
'549300LYFYVPUCG6SY25',
'549300NC3SZTETC10349',
'549300NEBDPH0ZXIF850',
'549300Z6OB1D4ZUBD145',
'63540061DPCBNMCGRY22',
'635400CE9HHFB55PEY43',
'635400GQWXFJDCQXW612',
'635400LNHEPZBRNB5D58',
'635400LRAHYBRUZCIH13',
'724500A11NIP5HCVF984',
'815600154F8F91CF6B05',
'8156002070DA4DCBFF31',
'81560027D07F9BDB8436',
'81560038903FF9FA8F80',
'815600522538355AE429',
'8156007395B20763EB44',
'8156009F40F6523F7022',
'815600A32DA05F693F24',
'8EFE15WY4PBBKG6GZI21',
'95980020140005184148',
'9598009X93GQBNHF1L85',
'959800T0J9ADL4GYSS41',
'9695000O0HNLFNR0FB30',
'969500DCEVPV6UIYK220',
'969500QLO3GN6GUB4P61',
'FI6C7E5PBUB3F9K43B44',
'GP5DT10VX1QRQUKVBK64',
'J4CP7MHCXR8DAQMKIL78',
'NNVPP80YIZGEY2314M97',
'RRAN7P32P0W0YY4XQW79',
'SI5RG2M0WQQLZCXKRM20'
)

In [12]:
bgroups = tuple(set(foreign_bg + euro_area_bg))

In [13]:
len(foreign_bg)

34

In [14]:
foreign_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(foreign_bg), 'lei'].unique())
euro_area_entity = tuple(df_mapping_cc.loc[df_mapping_cc['group_id_synth'].isin(euro_area_bg), 'lei'].unique())

# basic regression set-up

In [174]:
query = f"""

SELECT business_date, security_isin, lender_id as bank_id,
avg(repo_rate) as cleared_rate
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0 
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
AND repo_rate IS NOT NULL
GROUP BY business_date, security_isin, bank_id
ORDER BY business_date, security_isin, bank_id
  
"""

df_cleared = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_10252\557869383.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_cleared = pd.read_sql_query(query, cnxn)


In [175]:
query = f"""

SELECT business_date, security_isin, borrower_id as bank_id,
sum(nominal_value)/1e9 as intra_vol
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 1 
AND borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, security_isin, bank_id
ORDER BY business_date, security_isin, bank_id
  
"""

df_intra = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_10252\3106045254.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra = pd.read_sql_query(query, cnxn)


In [183]:
df = df_cleared.merge(df_intra, on = ['business_date', 'security_isin', 'bank_id'], how = 'left')

In [184]:
df['intra_vol'].fillna(0, inplace = True)

In [178]:
query = f"""

SELECT business_date, security_isin, borrower_id as bank_id,
sum(nominal_value)/1e9 as hf_vol
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0
AND borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
AND lender_country_residence = 'KY'
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, security_isin, bank_id
ORDER BY business_date, security_isin, bank_id
  
"""

df_hf = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_10252\571905444.py:29: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_hf = pd.read_sql_query(query, cnxn)


In [185]:
df = df.merge(df_hf, on = ['business_date', 'security_isin', 'bank_id'], how = 'left')

In [186]:
df['hf_vol'].fillna(0, inplace = True)

In [188]:
df = (
    df.groupby(['business_date', 'security_isin'], as_index=False)
      .agg(cleared_rate=('cleared_rate', 'mean'),
           intra_vol=('intra_vol', 'sum'),
           hf_vol=('hf_vol', 'sum'))
)

In [190]:
# Total intragroup at ISIN-day level (no bank dimension)
query_total_intra = f"""
SELECT business_date, security_isin,
sum(nominal_value)/1e9 as intra_vol_total
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 1
AND borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
AND central_clearing = 'non-cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
GROUP BY business_date, security_isin
"""
df_intra_total = pd.read_sql_query(query_total_intra, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_10252\138167014.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_intra_total = pd.read_sql_query(query_total_intra, cnxn)


In [191]:
# Merge with your existing df (which has linked intra_vol)
df = df.merge(df_intra_total, on=['business_date', 'security_isin'], how='left')
df['intra_vol_total'].fillna(0, inplace=True)

In [192]:
# Unlinked = total - linked
df['intra_vol_unlinked'] = df['intra_vol_total'] - df['intra_vol']

In [194]:
estr = pd.read_csv('ESTR.csv')

In [195]:
estr['Effective Date'] = pd.to_datetime(estr['DATE'], format='%d/%m/%Y')

In [196]:
df['business_date'] = pd.to_datetime(df['business_date'])

In [197]:
df = df.merge(estr[['Effective Date', 'ESTR']], left_on=['business_date'], right_on = ['Effective Date'], how = 'left')

In [199]:
df['special'] = (df['ESTR'] - df['cleared_rate'])*100

In [ ]:
# Bank benchmark. Cleared volume in the bond from banks doing neither chain
# nor HF activity that bond-day. Used as an activity control and placebo in
# the pricing regression. Merged into df before saving.
query_cleared_vol = f"""
SELECT business_date, security_isin, lender_id as bank_id,
sum(nominal_value)/1e9 as cleared_vol
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR'
AND intragroup = 0
AND central_clearing = 'cleared'
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC'
     AND s.security_isin IN {unique_govs})
  )
AND repo_rate IS NOT NULL
GROUP BY business_date, security_isin, bank_id
"""
df_cleared_vol = pd.read_sql_query(query_cleared_vol, cnxn)

# Remove the cleared activity of banks that are chain or HF active in that
# bond-day, so the benchmark is generic dealer activity only.
bench = df_cleared_vol.merge(
    df_intra[['business_date', 'security_isin', 'bank_id', 'intra_vol']],
    on=['business_date', 'security_isin', 'bank_id'], how='left'
).merge(
    df_hf[['business_date', 'security_isin', 'bank_id', 'hf_vol']],
    on=['business_date', 'security_isin', 'bank_id'], how='left'
)
bench['intra_vol'] = bench['intra_vol'].fillna(0)
bench['hf_vol'] = bench['hf_vol'].fillna(0)
bench = bench[(bench['intra_vol'] == 0) & (bench['hf_vol'] == 0)]

bench = (bench.groupby(['business_date', 'security_isin'], as_index=False)['cleared_vol'].sum()
              .rename(columns={'cleared_vol': 'bank_bench'}))
bench['business_date'] = pd.to_datetime(bench['business_date'])

df = df.merge(bench, on=['business_date', 'security_isin'], how='left')
df['bank_bench'] = df['bank_bench'].fillna(0)


In [ ]:
# Save the analysis dataset.
df.to_csv('intra_cleared.csv', index=False)


In [ ]:
# ============================================================
# CLEARED-MATCHED CHAIN AND HF MEASURES (appended, nothing above changed)
# Chain^in = min(intragroup borrowing, cleared lending) and its direct analog
# HF^in = min(HF leg, cleared lending), the matched gross volumes of Section 4.1,
# built at the bank-bond-day level. unlinked is the intragroup residual.
# Merges onto intra_cleared.csv (special, bank_bench) and saves
# intra_cleared_matched.csv. Self-contained, re-queries with fresh names.
# ============================================================
_coll = f"""
AND (
    (s.security_type = 'GOVS'
     AND LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.assttp_scty_issr_sector_riad = 'S1311')
    OR
    (LEFT(s.security_isin, 2) IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
     AND s.gnlcoll = 'SPEC' AND s.security_isin IN {unique_govs})
  )
"""

# Cleared lending (sourcing) volume per bank-bond-day
q_cl = f"""
SELECT business_date, security_isin, lender_id as bank_id, sum(nominal_value)/1e9 as cleared_lend
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 0 AND central_clearing = 'cleared'
{_coll}
AND repo_rate IS NOT NULL
GROUP BY business_date, security_isin, bank_id
"""
# Intragroup borrowing per bank-bond-day (euro area subsidiaries)
q_in = f"""
SELECT business_date, security_isin, borrower_id as bank_id, sum(nominal_value)/1e9 as intra_borrow
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 1 AND central_clearing = 'non-cleared'
AND borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
{_coll}
GROUP BY business_date, security_isin, bank_id
"""
# Direct hedge fund leg per bank-bond-day (euro area bank vs Cayman)
q_hf = f"""
SELECT business_date, security_isin, borrower_id as bank_id, sum(nominal_value)/1e9 as hf_borrow
FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s
WHERE s.business_date BETWEEN '2021-07-04' AND '2025-07-01'
AND nominal_ccy = 'EUR' AND intragroup = 0 AND central_clearing = 'non-cleared'
AND borrower_country_residence IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES')
AND lender_country_residence = 'KY'
{_coll}
GROUP BY business_date, security_isin, bank_id
"""
m_cl = pd.read_sql_query(q_cl, cnxn)
m_in = pd.read_sql_query(q_in, cnxn)
m_hf = pd.read_sql_query(q_hf, cnxn)

k = ['business_date', 'security_isin', 'bank_id']
m = m_cl.merge(m_in, on=k, how='outer').merge(m_hf, on=k, how='outer')
for c in ['cleared_lend', 'intra_borrow', 'hf_borrow']:
    m[c] = m[c].fillna(0)

# Cleared-matched volumes at the bank-bond-day level
m['chain_in'] = m[['intra_borrow', 'cleared_lend']].min(axis=1)
m['hf_in']    = m[['hf_borrow', 'cleared_lend']].min(axis=1)
m['unlinked'] = m['intra_borrow'] - m['chain_in']

# Collapse to ISIN-day
agg = m.groupby(['business_date', 'security_isin'], as_index=False).agg(
    intra_m=('chain_in', 'sum'),
    hf_m=('hf_in', 'sum'),
    unlinked_m=('unlinked', 'sum'))




C:\Users\hermesf\AppData\Local\Temp\ipykernel_8724\3598115949.py:51: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  m_cl = pd.read_sql_query(q_cl, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_8724\3598115949.py:52: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  m_in = pd.read_sql_query(q_in, cnxn)
C:\Users\hermesf\AppData\Local\Temp\ipykernel_8724\3598115949.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  m_hf = pd.read_sql_query(q_hf, cnxn)


FileNotFoundError: [Errno 2] No such file or directory: 'intra_cleared.csv'

In [16]:
# Merge onto the existing analysis dataset (carries special and bank_bench)
base = pd.read_csv('Data\\intra_cleared.csv')
base['business_date'] = pd.to_datetime(base['business_date'])
agg['business_date']  = pd.to_datetime(agg['business_date'])
out = base.merge(agg, on=['business_date', 'security_isin'], how='left')
for c in ['intra_m', 'hf_m', 'unlinked_m']:
    out[c] = out[c].fillna(0)
out.to_csv('Data\\intra_cleared_matched.csv', index=False)

In [9]:
df = pd.read_csv('Data\\intra_cleared_matched.csv')

In [10]:
df.head()

,business_date,security_isin,cleared_rate,intra_vol,hf_vol,intra_vol_total,intra_vol_unlinked,Effective Date,ESTR,special,bank_bench,intra_m,hf_m,unlinked_m
0,2021-07-05,AT0000383864,-0.625963,0.000000,0.0,0.027636,0.027636,2021-07-05,-0.566,5.996296,0.227094,0.000000,0.0,0.027636
1,2021-07-05,AT0000A001X2,-0.598793,0.000000,0.0,0.166934,0.166934,2021-07-05,-0.566,3.279339,0.388625,0.000000,0.0,0.166934
2,2021-07-05,AT0000A04967,-0.649782,0.137901,0.0,0.362546,0.224645,2021-07-05,-0.566,8.378205,1.775271,0.137901,0.0,0.224645
3,2021-07-05,AT0000A0DXC2,-0.600113,0.031871,0.0,0.035032,0.003161,2021-07-05,-0.566,3.411255,0.553343,0.031871,0.0,0.003160
4,2021-07-05,AT0000A0N9A0,-0.586905,0.000000,0.0,0.014489,0.014489,2021-07-05,-0.566,2.090476,0.105229,0.000000,0.0,0.014488


In [12]:
df['intra_m'].mean()

0.1260229472327588

In [13]:
df['intra_m'].quantile(0.99)

1.1634898399999998

In [14]:
df['hf_m'].mean()

0.14990886854515773